In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([39, 40, 44, 23, 51, 50, 35, 79, 55, 32, 32, 32, 30, 44, 23])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.zeros((arr.size, label), dtype=np.float32)
    one_hot[np.arange(one_hot.shape[0]), arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [5]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [6]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [7]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[39 40 44 23 51 50 35 79 55 32]
 [34 56 61 79 51 40 44 51 79 44]
 [50 61 69 79 56 35 79 44 79 26]
 [34 79 51 40 50 79 59 40 27 50]
 [79 34 44 65 79 40 50 35 79 51]
 [59 10 34 34 27 56 61 79 44 61]
 [79 62 61 61 44 79 40 44 69 79]
 [22 15 66 56 61 34 73 74 42 79]]
y
 [[40 44 23 51 50 35 79 55 32 32]
 [56 61 79 51 40 44 51 79 44 51]
 [61 69 79 56 35 79 44 79 26 56]
 [79 51 40 50 79 59 40 27 50 26]
 [34 44 65 79 40 50 35 79 51 50]
 [10 34 34 27 56 61 79 44 61 69]
 [62 61 61 44 79 40 44 69 79 34]
 [15 66 56 61 34 73 74 42 79 49]]


In [8]:
class CharNN(nn.Module):    
    def __init__(self, tokens, n_layer=2, n_hidden=256, drop_prob=0.5, lr=0.001):
        super().__init__()

        self.n_layer = n_layer
        self.n_hidden = n_hidden
        self.lr = lr

        self.chars = tokens
        self.int_char = dict(enumerate(self.chars))
        self.char_int = {ch : i for i, ch in self.int_char.items()}

        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layer, dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)

        self.fc = nn.Linear(n_hidden, len(self.chars))

    def forward(self, x, hidden):
        r_output, hidden = self.lstm(x, hidden)

        out = self.dropout(r_output)
       
        out = out.contiguous().view(-1, self.n_hidden)

        out = self.fc(out)

        return out, hidden
    
    def init_hidden (self, batch_size):
        
        weight = next(self.parameters()).data
        
        hidden = (weight.new(self.n_layer, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layer, batch_size, self.n_hidden).zero_())
        
        return hidden

In [ ]:
def train(net, data, epoch=10, batch_size=10, seq_len=50, lr=0.001, clip=5, val_frac=0.1):

    net.train()

    opt = torch.optim.Adam(net.parameters(), lr=lr)
    crietrion = nn.CrossEntropyLoss()

    val_idx = int(len(data)*(1-val_frac))
    data, val_data = data[:val_idx], data[val_idx:]

    n_chars = len(net.chars)

    for n in range(epoch):
        h = net.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_len):
            x = one_hot_encode(x, n_chars)
            input, target = torch.from_numpy(x), torch.from_numpy(y)

            h = tuple([each.data for each in h])

            net.zero_grad()
            output, h = net(input, h)
            loss = crietrion(output, target.view(batch_size*seq_len).long())

            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        
        else:
            val_losses = []
            net.eval()
            val_h = net.init_hidden(batch_size)

            for x, y in get_batches(val_data, batch_size, seq_len):
                x = one_hot_encode(x,n_chars)
                inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

                val_h = tuple([each.data for each in val_h])

                output, val_h = net(inputs, val_h)
                val_loss = crietrion(output, targets.view(batch_size*seq_len).long())

                val_losses.append(val_loss.item())

            net.train()

            print(f"Epoch: {n+1}, Train loss: {loss.item():.4f}, Val loss: {np.mean(val_losses):.4f}")

In [10]:
n_hidden = 512
n_layer = 2

net = CharNN(chars, n_layer=n_layer, n_hidden=n_hidden)
print(net)

CharNN(
  (lstm): LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=83, bias=True)
)


In [11]:
batch_size = 128
seq_len = 100
epoch = 10

train(net, encoded, epoch, batch_size, seq_len)

Epoch: 1, Train loss: 3.097653, Val loss: 3.111702
Epoch: 2, Train loss: 2.749155, Val loss: 2.672285
Epoch: 3, Train loss: 2.510003, Val loss: 2.415669
Epoch: 4, Train loss: 2.386825, Val loss: 2.261676
Epoch: 5, Train loss: 2.310861, Val loss: 2.169334
Epoch: 6, Train loss: 2.244852, Val loss: 2.105954
Epoch: 7, Train loss: 2.183521, Val loss: 2.044739
Epoch: 8, Train loss: 2.155560, Val loss: 2.012150
Epoch: 9, Train loss: 2.112190, Val loss: 1.969110
Epoch: 10, Train loss: 2.065028, Val loss: 1.934941


In [11]:
model_name = 'CheckPoint_10.pth'

checkpoint = {
    'n_hidden': net.n_hidden,
    'n_layer': net.n_layer,
    'state_dict': net.state_dict(),
    'chars': net.chars
}

with open(model_name, 'wb') as f:
    torch.save(checkpoint, f)

In [15]:
with open('CheckPoint_10.pth', 'rb') as f:
	checkpoint = torch.load(f)

loaded = CharNN(tokens=checkpoint['chars'], n_hidden=checkpoint['n_hidden'], n_layer=checkpoint['n_layer'])
loaded.load_state_dict(checkpoint['state_dict'])
print(loaded)

CharNN(
  (lstm): LSTM(83, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=83, bias=True)
)


C:\Users\Yonatan\AppData\Local\Temp\ipykernel_7756\933813931.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(f)


In [22]:
def Predict(net, char, h=None, top_k=None):
    x = np.array([[net.char_int[char]]])
    x = one_hot_encode(x, len(net.chars))
    inputs = torch.from_numpy(x)

    h = tuple([one for one in h])

    out, h = net(inputs, h)
    p = F.softmax(out, dim=1).data

    if top_k is None:
        top_ch = np.arange(len(net.chars))
    else:
        p, top_ch = p.topk(top_k)
        top_ch = top_ch.numpy().squeeze()

    p = p.numpy().squeeze()
    char = np.random.choice(top_ch, p=p/p.sum())

    return net.int_char[char], h

In [23]:
def Sample(net, size, prime='The',top_k=None):
    
    net.eval()

    chars = [ch for ch in prime]
    h = net.init_hidden(1)

    for ch in chars:
        char, h = Predict(net, ch, h, top_k=top_k)
    
    chars.append(char)

    for n in range(size):
        char, h = Predict(net, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

In [25]:
print(Sample(loaded, 1000, 'The', top_k=5))

The`rw*tr`*?`*tr*`r`rt`r*``rtt*?rr`tr?**rw**?*tt*r*w**`??`r`?ttt*t*rww`t?`r`t??*```r``*?``r?`ttt**rwwr*?rt**??r`trt??tt?`*t`t?t`?r*?`?`**`t?``*rr*?*t?tt*tr`t*rr?t*`??*`r``r`tt*`?``**r?`*``rt??t?t?rww??`*``trtr`*tt?*t?tt*?t?t`r?`??*rrttt*rw?r*?r``*````?r*`rrrt`*?*rt*r***`?`rr?ttr`t*trt`t*`tt`*?*`?rt??*?t`???*r```*r*r??r`*??rtt**t???*trw*t?*`t?t???`?`t?t?rtr?*`tr`t?rr**??**?r?`?`rr?r`?t*?t?**?t`r**t?rr?`tt*?t?`??`trrtr``tr``?r`*``??r``*`?`*r*`?`trt``*`r*?**t`*t*r?**?*`t`r`?r*r?t?`r`**rtt?t*t?`tttrtrt`??tr*`?*`tttt*``*`ttr?*``*r?t?rttr*w?ttrttt```??r`r???rw?t*rt??`r`t`rr???`ttrrt?*?t``*`r*r*t**tt*trr`wr`rt??````*t`r?t*?r``t`r**ttt`rtr```t?*r`t*`???rr*t?*r?t*??*??r`t?trr?`wr**r*t?*rwt**??*?`tt*ttt`*`r**rtr`trr*?*`**`?rt**rww?r`**t*?r?r**?tr***t?r`?`?`ttrt`rrw*?`r?*?r`*t`t*r*?`?`??`r*`t``*trtr?`**rrww?**?*r?`?tt*?*t?`t`t`t?``*r?t`?r`*``?ttr`t?r*r`wrr??``t**r?rt`?t`rrwwr?`*??*rtt*r`????*?t??t`t??`trtt`r?t?tttrtw**?t*`*??`?r?`t**t`*t*trr?*`tttrr?t?tr``t*t*????rww`t`??trt*??*t*rw???t??r*t*?``?